## **Schema Inspection — All 8 Bronze Files**

In [ ]:
bronze_path = "abfss://246f1c03-56af-4f6f-8640-aff9681c241b@onelake.dfs.fabric.microsoft.com/656f0bb9-d036-4f99-9e67-a4d0155df259/Files/Raw_Data_Bronze"

tables = ["currencyexchange", "customer", "date", "orderrows", 
          "orders", "product", "sales", "store"]

for table in tables:
    print(f"\n{'='*50}")
    print(f"TABLE: {table}")
    print('='*50)
    df = spark.read.format("delta").load(f"{bronze_path}/{table}")
    df.printSchema()
    print(f"Row count: {df.count()}")
    print('='*50)
    print("Succesfully Inspected")

## **Registered the Table inside the Views**

In [ ]:
bronze_path = "abfss://246f1c03-56af-4f6f-8640-aff9681c241b@onelake.dfs.fabric.microsoft.com/656f0bb9-d036-4f99-9e67-a4d0155df259/Files/Raw_Data_Bronze"

tables = ["currencyexchange", "customer", "date", "orderrows", 
          "orders", "product", "sales", "store"]

for table in tables:
    df = spark.read.format("delta").load(f"{bronze_path}/{table}")
    df.createOrReplaceTempView(f"bronze_{table}")
    print(f"✅ Registered: bronze_{table}")

## **Explore the Currency Exchange table**

In [ ]:
# Explore the table content
display(spark.sql("SELECT * FROM bronze_currencyexchange LIMIT 10"))

# Row count
spark.sql("SELECT COUNT(*) AS TotalRows FROM bronze_currencyexchange").show()

## **Silver: currencyexchange - Transformation**

In [ ]:
silver_currencyexchange = spark.sql("""
    SELECT
        CAST(Date AS DATE)        AS Date,
        FromCurrency,
        ToCurrency,
        CAST(Exchange AS DOUBLE)  AS ExchangeRate
    FROM bronze_currencyexchange
    WHERE Date IS NOT NULL
      AND Exchange IS NOT NULL
""")

silver_currencyexchange.write.format("delta").mode("overwrite").saveAsTable("silver_currencyexchange")
print(f"✅ silver_currencyexchange written — {silver_currencyexchange.count()} rows")

## **Explore the Customer table**

In [ ]:
# Explore the table content
display(spark.sql("SELECT * FROM bronze_customer LIMIT 10"))

# Row count
spark.sql("SELECT COUNT(*) AS TotalRows FROM bronze_customer").show()

# Print all column names in SQL-ready format for transformation
cols = spark.sql("SELECT * FROM bronze_customer LIMIT 1").columns
print(",\n".join(cols))

## **Silver: Customer - Transformation**

In [ ]:
silver_customer = spark.sql("""
    SELECT
        CustomerKey,
        GeoAreaKey,
        CAST(StartDT AS DATE) AS StartDate,
        CAST(EndDT AS DATE) AS EndDate,
        Continent,
        INITCAP(Gender) AS Gender,
        Title,
        CONCAT(GivenName, ' ',
               COALESCE(MiddleInitial, ''), ' ',
               Surname) AS FullName,
        StreetAddress,
        City,
        State,
        StateFull,
        ZipCode,
        Country,
        CountryFull,
        CAST(Birthday AS DATE) AS Birthday,
        Age,
        Occupation,
        Company
    FROM bronze_customer
    WHERE CustomerKey IS NOT NULL
""")

silver_customer.write.format("delta").mode("overwrite").saveAsTable("silver_customer")
print(f"✅ silver_customer written — {silver_customer.count()} rows")

## **Explore the Date table**

In [ ]:
# Explore the table content
display(spark.sql("SELECT * FROM bronze_date LIMIT 10"))

# Row count
spark.sql("SELECT COUNT(*) AS TotalRows FROM bronze_date").show()

# Print all column names in SQL-ready format for transformation
cols = spark.sql("SELECT * FROM bronze_date LIMIT 1").columns
print(",\n".join(cols))

## **Silver: Date - Transformation**

In [ ]:
silver_date = spark.sql("""
    SELECT
        CAST(Date AS DATE) AS Date,
        CAST(DateKey AS INTEGER) AS DateKey,
        Year,
        YearQuarter,
        YearQuarterNumber,
        Quarter,
        YearMonth,
        YearMonthShort,
        YearMonthNumber,
        Month,
        MonthShort,
        MonthNumber,
        DayofWeek,
        DayofWeekShort,
        DayofWeekNumber,
        WorkingDay,
        WorkingDayNumber
    FROM bronze_date
    WHERE Date IS NOT NULL
""")

silver_date.write.format("delta").mode("overwrite").saveAsTable("silver_date")
print(f"✅ silver_date written — {silver_date.count()} rows")

## **Explore the Orderrows table**

In [ ]:
# Explore the table content
display(spark.sql("SELECT * FROM bronze_orderrows LIMIT 10"))

# Row count
spark.sql("SELECT COUNT(*) AS TotalRows FROM bronze_orderrows").show()

# Print all column names in SQL-ready format for transformation
cols = spark.sql("SELECT * FROM bronze_orderrows LIMIT 1").columns
print(",\n".join(cols))

## **Silver: Orderrows - Transformation**

In [ ]:
silver_orderrows = spark.sql("""
    SELECT
        OrderKey,
        RowNumber,
        ProductKey,
        Quantity,
        CAST(UnitPrice AS DOUBLE)  AS UnitPrice,
        CAST(NetPrice AS DOUBLE)   AS NetPrice,
        CAST(UnitCost AS DOUBLE)   AS UnitCost
    FROM bronze_orderrows
    WHERE OrderKey IS NOT NULL
      AND Quantity IS NOT NULL
""")

silver_orderrows.write.format("delta").mode("overwrite").saveAsTable("silver_orderrows")
print(f"✅ silver_orderrows written — {silver_orderrows.count()} rows")

## **Explore the Orders table**

In [ ]:
# Explore the table content
display(spark.sql("SELECT * FROM bronze_orders LIMIT 10"))

# Row count
spark.sql("SELECT COUNT(*) AS TotalRows FROM bronze_orders").show()

# Print all column names in SQL-ready format for transformation
cols = spark.sql("SELECT * FROM bronze_orders LIMIT 1").columns
print(",\n".join(cols))

## **Silver: Order - Transformation**

In [ ]:
silver_orders = spark.sql("""
    SELECT
        OrderKey,
        CustomerKey,
        StoreKey,
        CAST(DT AS DATE) AS OrderDate,
        CAST(DeliveryDate AS DATE) AS DeliveryDate,
        CurrencyCode
    FROM bronze_orders
    WHERE OrderKey IS NOT NULL
      AND CustomerKey IS NOT NULL
""")

silver_orders.write.format("delta").mode("overwrite").saveAsTable("silver_orders")
print(f"✅ silver_orders written — {silver_orders.count()} rows")

## **Explore the Product table**

In [ ]:
# Explore the table content
display(spark.sql("SELECT * FROM bronze_product LIMIT 10"))

# Row count
spark.sql("SELECT COUNT(*) AS TotalRows FROM bronze_product").show()

# Print all column names in SQL-ready format for transformation
cols = spark.sql("SELECT * FROM bronze_product LIMIT 1").columns
print(",\n".join(cols))

## **Silver: Product - Transformation**

In [ ]:
silver_product = spark.sql("""
    SELECT
        ProductKey,
        ProductCode,
        ProductName,
        Manufacturer,
        Brand,
        Color,
        WeightUnit,
        CAST(Weight AS DOUBLE)  AS Weight,
        CAST(Cost AS DOUBLE)    AS Cost,
        CAST(Price AS DOUBLE)   AS Price,
        CategoryKey,
        CategoryName,
        SubCategoryKey,
        SubCategoryName
    FROM bronze_product
    WHERE ProductKey IS NOT NULL
""")

silver_product.write.format("delta").mode("overwrite").saveAsTable("silver_product")
print(f"✅ silver_product written — {silver_product.count()} rows")

## **Explore the Store table**

In [ ]:
# Explore the table content
display(spark.sql("SELECT * FROM bronze_store LIMIT 10"))

# Row count
spark.sql("SELECT COUNT(*) AS TotalRows FROM bronze_store").show()

# Print all column names in SQL-ready format for transformation
cols = spark.sql("SELECT * FROM bronze_store LIMIT 1").columns
print(",\n".join(cols))

## **Silver: Store - Transformation**

In [ ]:
silver_store = spark.sql("""
    SELECT
        StoreKey,
        StoreCode,
        GeoAreaKey,
        CountryCode,
        CountryName,
        State,
        CAST(OpenDate AS DATE)  AS OpenDate,
        CAST(CloseDate AS DATE) AS CloseDate,
        Description,
        SquareMeters,
        Status
    FROM bronze_store
    WHERE StoreKey IS NOT NULL
""")

silver_store.write.format("delta").mode("overwrite").saveAsTable("silver_store")
print(f"✅ silver_store written — {silver_store.count()} rows")

## **Explore the Sales table**

In [ ]:
# Explore the table content
display(spark.sql("SELECT * FROM bronze_sales LIMIT 10"))

# Row count
spark.sql("SELECT COUNT(*) AS TotalRows FROM bronze_sales").show()

# Print all column names in SQL-ready format for transformation
cols = spark.sql("SELECT * FROM bronze_sales LIMIT 1").columns
print(",\n".join(cols))

## **Silver: Sales - Transformation**

In [ ]:
silver_sales = spark.sql("""
    SELECT
        OrderKey,
        LineNumber,
        CAST(OrderDate AS DATE)       AS OrderDate,
        CAST(DeliveryDate AS DATE)    AS DeliveryDate,
        CustomerKey,
        StoreKey,
        ProductKey,
        Quantity,
        CAST(UnitPrice AS DOUBLE)     AS UnitPrice,
        CAST(NetPrice AS DOUBLE)      AS NetPrice,
        CAST(UnitCost AS DOUBLE)      AS UnitCost,
        CurrencyCode,
        CAST(ExchangeRate AS DOUBLE)  AS ExchangeRate,
        ROUND(Quantity * CAST(UnitPrice AS DOUBLE), 2)    AS SalesAmount,
        ROUND((Quantity * CAST(UnitPrice AS DOUBLE)) -
              (Quantity * CAST(UnitCost AS DOUBLE)), 2)   AS GrossProfit
    FROM bronze_sales
    WHERE Quantity IS NOT NULL
      AND UnitPrice IS NOT NULL
      AND UnitCost IS NOT NULL
""")

silver_sales.write.format("delta").mode("overwrite").saveAsTable("silver_sales")
print(f"✅ silver_sales written — {silver_sales.count()} rows")